# Lab 3.1: Getting Started with Strands Agents

This notebook introduces the core concepts of [Strands Agents](https://strandsagents.com) that you will use throughout the workshop:

1. Creating an agent
2. Choosing a model provider
3. Building tools with `@tool`
4. Using lifecycle hooks
5. Assembling `hotel_agent`

Sections 1 through 4 each build their own small agent to make one point. Section 5 puts three of those pieces together into `hotel_agent`, the one agent Labs 4 and 5 carry forward.

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# At an AWS Event: dependencies are pre-installed. Run this cell as-is.
# Self-paced (outside an AWS event): uncomment the line below first.
# ─────────────────────────────────────────────────────────────────────
# !pip install -r requirements.txt

import os

import boto3
from dotenv import load_dotenv

from workshop.bedrock_providers import default_model_id

load_dotenv()

# The model id every agent below is pinned to. Strands has its own default
# model, and an Agent built without a model quietly runs that one instead of
# the model the rest of the workshop uses, so bind it once here and pass it to
# every Agent in this notebook. The id itself has exactly one definition, in
# workshop/src/workshop/bedrock_providers.py; a MODEL_ID in the environment
# still overrides it.
MODEL_ID = default_model_id()

# Every agent turn below calls Amazon Bedrock, and the two sections that use the
# real retriever also read the graph Lab 1 built. Both are checked once here, so
# an unconfigured environment skips those cells instead of raising in each one.
AGENT_READY = boto3.Session().get_credentials() is not None
NEO4J_VARS = ("NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD")
GRAPH_READY = AGENT_READY and all(os.environ.get(name) for name in NEO4J_VARS)

print("✅ Environment ready" if AGENT_READY else "No AWS credentials found.")
print(f"Model: {MODEL_ID}")
if AGENT_READY and not GRAPH_READY:
    print("Neo4j is not configured, so the grounded-retrieval cells will skip.")

---
## 1. Creating an Agent

A Strands agent combines a large language model (LLM) with tools. The agent uses the LLM to reason about the user's request and decide which tools to call.

By default, Strands uses **Amazon Bedrock** as the model provider. No extra configuration is needed once your AWS credentials are set. The model itself is a different matter: leave `model` off and Strands picks its own default model id, so every agent here passes the `MODEL_ID` bound in the setup cell.

In [ ]:
from strands import Agent

# Create an agent with a system prompt.
# The system prompt defines the agent's personality and instructions.
agent_travel = Agent(
    model=MODEL_ID,
    system_prompt="You are a helpful travel assistant. Answer questions about hotels and travel.",
)

# Talk to the agent. It runs on Amazon Bedrock, on the model id pinned above.
# Strands streams the answer itself, through the default PrintingCallbackHandler,
# so wrapping this call in print() would show the whole answer a second time.
if AGENT_READY:
    agent_travel("What should I consider when booking a hotel in Lisbon?")
else:
    print("Skipping: no AWS credentials.")

---
## 2. Model Providers

Strands supports multiple model providers. This workshop uses Amazon Bedrock, but you can switch to others.

There are two ways to pin the Bedrock model. Section 1 used the first: pass the model id as a string, and Strands builds a Bedrock model around it. The cell below uses the second: build the `BedrockModel` yourself. Reach for that form once something beyond the id has to be set, such as the region, the temperature, or a boto3 session. Labs 4 and 5 both build the model object, because both pin a region on it explicitly.

See all providers: [Strands Model Providers](https://strandsagents.com/docs/user-guide/concepts/model-providers/amazon-bedrock/)

In [ ]:
from strands import Agent
from strands.models import BedrockModel

# Section 1 already showed the string form, Agent(model=MODEL_ID, ...). This is
# the object form, built from the same pinned id, and it is the one Labs 4 and 5
# use so they can name a region on the model itself.
agent_specific = Agent(
    model=BedrockModel(model_id=MODEL_ID),
    system_prompt="You are a helpful assistant.",
)

# As above, Strands streams the answer, so there is no print() wrapper here.
if AGENT_READY:
    agent_specific("Say hello in one sentence.")
else:
    print("Skipping: no AWS credentials.")

---
## 3. Creating Tools

Tools are Python functions decorated with `@tool`. The agent reads the function name and docstring to decide when to call each tool.

**Docstrings are critical.** The agent uses them to match user queries to tools. A vague docstring leads to wrong tool selection.

Two of the three tools below return invented strings, which is all a primer needs to show selection working. The third is the real hybrid retriever from Lab 2, imported rather than rewritten. Watching the agent choose between them is the whole point of the section.

### The third tool takes one field, and that is the design

`search_hotel_knowledge` is the frozen one-field contract from the shared `workshop` package. It runs the `HybridCypherRetriever` over the vector and full-text indexes, applies the one reviewed Cypher traversal, and returns bounded JSON facts rather than prose: chunk evidence, the hybrid score, the query terms found verbatim in that evidence, and the hotel's stable `hotel_id`, name, address, guest rating, and amenities.

The `@tool` wrapper exposes `query` and nothing else. The model cannot pick a retriever, select a ranker, set an alpha, or ask for more results. That ceiling is deliberate. Every extra field on a tool signature moves a decision from the engineers who reviewed the retrieval path to a model that will improvise one at run time, and a retrieval strategy chosen per call is a retrieval strategy nobody can reproduce when an answer comes out wrong. Holding the signature at one field keeps the question variable and the method fixed, so the same query returns the same evidence tomorrow.

The narrow boundary is also what makes the tool portable. Lab 4 rebuilds the agent and Lab 5 deploys it to AgentCore Runtime, and neither one touches this function, because there is no knob on it to retune.

See: [Strands Tools Documentation](https://strandsagents.com/docs/user-guide/concepts/tools/custom-tools/)

In [ ]:
import json

from strands import Agent, tool

from workshop.hybrid_retrieval import search_hotel_knowledge

@tool
def search_hotels(city: str, max_price: int = 500) -> str:
    """Search for available hotels in a city under a maximum price per night."""
    # In a real application, this would query a database.
    # For this demo, we return a simulated response.
    return f"Found 3 hotels in {city} under ${max_price}/night:\n" \
           f"  1. AnyCompany {city} Resort ($95/night)\n" \
           f"  2. AnyCompany {city} Central ($110/night)\n" \
           f"  3. AnyCompany {city} Budget ($65/night)"

@tool
def book_hotel(hotel_name: str, guest_name: str, nights: int = 1) -> str:
    """Book a hotel room for a guest. Returns a booking confirmation ID."""
    total = nights * 95  # Simulated price
    return f"Booking confirmed: {guest_name} at {hotel_name}\n" \
           f"  {nights} night(s), total: ${total}\n" \
           f"  Confirmation ID: BK-001"

@tool
def search_hotel_knowledge_tool(query: str) -> str:
    """Look up amenities, ratings, and policies for a specific named hotel."""
    # The one tool here that is not simulated. It runs Lab 2's hybrid retrieval
    # against the graph and returns bounded JSON facts, not prose.
    return json.dumps(search_hotel_knowledge(query), ensure_ascii=False)

# Create an agent with tools
tools = [search_hotels, book_hotel, search_hotel_knowledge_tool]
agent_tools = Agent(
    model=MODEL_ID,
    tools=tools,
    system_prompt="You are a hotel booking assistant.",
)

print("Agent created with 3 tools:", [t.__name__ for t in tools])

In [ ]:
# The agent decides which tool to call based on the query
print("=== Test 1: Hotel search ===")
if AGENT_READY:
    agent_tools("Find me hotels in Lisbon under $100")
else:
    print("Skipping: no AWS credentials.")

# You should see the agent call search_hotels and return the results

In [ ]:
print("=== Test 2: Grounded hotel knowledge (different tool) ===")
if GRAPH_READY:
    agent_tools("What amenities does AnyCompany Cairo Nile View have?")
else:
    print("Skipping: needs AWS credentials and the graph Lab 1 built.")

# The agent should call search_hotel_knowledge_tool, NOT search_hotels.
# Compare the answer with Test 1: these amenities came out of the graph.

In [ ]:
print("=== Test 3: Booking ===")
if AGENT_READY:
    agent_tools("Book AnyCompany Lisbon Resort for Alice for 3 nights")
else:
    print("Skipping: no AWS credentials.")

# The agent should call book_hotel with the correct parameters

---
## 4. Lifecycle Hooks

Hooks intercept the agent's execution at specific points. The most important hook for this workshop is `BeforeToolCallEvent`. It fires after the LLM decides to call a tool and **before** the tool executes.

Setting `event.cancel_tool` prevents the tool from executing. The LLM receives the cancellation message instead of the tool result. **The LLM cannot bypass this.** It happens at the framework level.

Note where the number lives: the limit of 10 is a Python literal inside `MaxGuestsHook` below, sitting in the same file as the agent that enforces it. Lab 4 comes back to that.

See: [Strands Hooks Documentation](https://strandsagents.com/docs/user-guide/concepts/agents/hooks/)

In [ ]:
from strands import Agent, tool
from strands.hooks import HookProvider, HookRegistry
from strands.hooks.events import BeforeToolCallEvent

@tool
def book_room(hotel: str, guests: int = 1) -> str:
    """Book a hotel room for a number of guests."""
    return f"SUCCESS: Booked {hotel} for {guests} guests"

class MaxGuestsHook(HookProvider):
    """Block bookings with more than 10 guests."""

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeToolCallEvent, self.check)

    def check(self, event: BeforeToolCallEvent) -> None:
        if event.tool_use["name"] == "book_room":
            guests = event.tool_use["input"].get("guests", 1)
            if guests > 10:
                # This BLOCKS the tool call. The LLM cannot override it.
                event.cancel_tool = f"BLOCKED: {guests} guests exceeds maximum of 10"

# Agent WITHOUT hook: no protection
agent_no_hook = Agent(
    model=MODEL_ID,
    tools=[book_room],
    system_prompt="You are a booking assistant.",
)

# Agent WITH hook: enforces the rule. Same model, same tool, one difference.
agent_with_hook = Agent(
    model=MODEL_ID,
    tools=[book_room],
    hooks=[MaxGuestsHook()],
    system_prompt="You are a booking assistant.",
)

print("Created two agents: one without hook, one with MaxGuestsHook")

In [ ]:
print("=== Without hook: books 15 guests (no protection) ===")
if AGENT_READY:
    agent_no_hook("Book AnyCompany Lisbon Resort for 15 guests")
else:
    print("Skipping: no AWS credentials.")

# The agent books 15 guests: no validation

In [ ]:
print("=== With hook: BLOCKS 15 guests ===")
if AGENT_READY:
    agent_with_hook("Book AnyCompany Lisbon Resort for 15 guests")
else:
    print("Skipping: no AWS credentials.")

# The hook blocks the call. The agent reports the error to the user.

---
## 5. Assembling `hotel_agent`

The four sections above each built their own agent to make one point. This
section builds the one agent the rest of the workshop uses, and it takes exactly
three pieces from what you just saw: the real retriever tool from section 3, and
the `book_room` tool and the guest-limit hook from section 4.

Nothing else carries forward. `search_hotels` and `book_hotel` invented their
answers, which was fine for showing tool selection and is not fine once an agent
is answering about real hotels.

Then ask it two questions, one it can answer from the graph and one the hook has
to stop.

In [ ]:
if not AGENT_READY:
    print("Skipping: no AWS credentials.")
else:
    from workshop.hybrid_retrieval import GROUNDING_INSTRUCTIONS

    hotel_agent = Agent(
        name="hotel_agent",
        model=MODEL_ID,
        tools=[search_hotel_knowledge_tool, book_room],
        hooks=[MaxGuestsHook()],
        system_prompt=(
            "You are a hotel assistant. Call search_hotel_knowledge_tool before "
            "answering any question about a hotel, and take bookings only "
            "through book_room.\n\n" + GROUNDING_INSTRUCTIONS
        ),
    )
    print("hotel_agent created with 2 tools and MaxGuestsHook attached")

In [ ]:
print("=== A question the graph can answer ===")
if GRAPH_READY:
    hotel_agent(
        "What amenities and guest rating does AnyCompany Cairo Nile View have?"
    )
else:
    print("Skipping: needs AWS credentials and the graph Lab 1 built.")

In [ ]:
print("=== A booking the hook has to stop ===")
if GRAPH_READY:
    hotel_agent("Book AnyCompany Cairo Nile View for 15 guests")
else:
    print("Skipping: needs AWS credentials and the graph Lab 1 built.")

# hotel_agent is instructed to look a hotel up before it books, so this turn
# reaches the graph as well as Bedrock and is gated the same way the cell above
# it is. The hook cancels the tool call before book_room runs. Lab 4 replaces
# that Python literal with a rule read from the graph.

---
## Summary

| Concept | What it does | Where the workshop uses it |
|---------|-------------|----------------|
| `Agent` + `@tool` | LLM reasons and calls functions | Labs 3, 4, 5, 6 |
| Model providers | Choose Bedrock, Anthropic, OpenAI, etc. | Every lab that calls a model |
| The one-field tool signature | Keeps the retrieval method with the engineers instead of the model | Labs 3, 4, 5 |
| `BeforeToolCallEvent` + `cancel_tool` | Block tool calls that violate rules | Lab 4 |

Every agent above was built with an explicit `model`, so the whole notebook runs on the one model id the workshop standardizes on.

`hotel_agent` is now assembled, carrying the real retriever tool, `book_room`, and `MaxGuestsHook`. Lab 4 rebuilds an agent under the same name, adds the idempotent reservation write as `create_reservation_request_tool`, and drops `MaxGuestsHook` because the limit it enforced moves into the graph.